# Insurance Charges — GPU Training (Colab)

Trains the same model this repo trains locally (`XGBoost` on the insurance-charges
tabular dataset), but on a **Colab GPU runtime**, and writes back artifacts in the
exact folder layout the local project expects (`artifacts/preprocessing/`,
`artifacts/models/`, `artifacts/evaluation/`) so you can download the zip and drop
it straight into your project root.

**Before running:** `Runtime → Change runtime type → T4 GPU` (or better).

### Honest take on GPU vs. CPU for this dataset first
This is a **tabular** regression problem (~100k rows, 11 features), not a
deep-learning workload:
- `LinearRegression` / `Ridge` have **no GPU code path at all** (scikit-learn)
  — they only ever run on CPU, and they train in ~10 seconds either way.
- `RandomForest` also has **no GPU support** in scikit-learn — its cost comes
  from tree count × depth, not something a GPU accelerates.
- **XGBoost is the only model here that can use a GPU** (`device="cuda"`,
  `tree_method="hist"`), and it's genuinely worth it once you're doing a
  wide hyperparameter search or training on the full ~1,000,000-row source
  dataset instead of the 100k working sample.
- In the artifacts already in this repo, plain `LinearRegression`
  (val R² = 0.9957, ~10s to train) actually **beat** tuned XGBoost
  (val R² = 0.9954) and crushed `RandomForest` (val R² = 0.957, ~85 minutes
  on CPU). That strongly suggests the target is close to linear in the
  engineered features — so don't expect GPU XGBoost to beat the linear
  baseline on *accuracy*. What GPU buys you here is **speed**: it lets you
  run a much larger `RandomizedSearchCV` / train on the full dataset in
  minutes instead of hours.


## 1. Environment setup

In [ ]:
!nvidia-smi


In [ ]:
!pip install -q "xgboost>=2.1.0" "scikit-learn>=1.5.1" "pandas>=2.2.2" "PyYAML>=6.0.1" "joblib>=1.4.2" "matplotlib>=3.9.2" "seaborn>=0.13.2"

import xgboost as xgb
print("xgboost:", xgb.__version__)

# Quick GPU sanity check for XGBoost specifically (separate from nvidia-smi —
# confirms this xgboost *build* actually has GPU support compiled in).
import numpy as np
try:
    probe = xgb.XGBRegressor(n_estimators=1, max_depth=1, device="cuda", tree_method="hist")
    probe.fit(np.zeros((2, 1)), np.zeros(2))
    GPU_OK = True
    print("XGBoost GPU support: OK — will train with device='cuda'")
except Exception as e:
    GPU_OK = False
    print("XGBoost GPU support NOT available, falling back to CPU. Reason:", e)


## 2. Load data

Two options — use whichever is easiest:

**Option A — upload the CSVs directly** (fastest for this dataset's size, ~7–10MB each).
Run the cell below and pick `train.csv`, `val.csv`, `test.csv`, `raw.csv` from
`artifacts/data/` in your local project.

**Option B — mount Google Drive** if you've already uploaded the `DAMO/` project
folder there — uncomment the Drive cell instead and adjust the path.


In [ ]:
# --- Option A: direct upload ---
from google.colab import files
import shutil, os

os.makedirs("artifacts/data", exist_ok=True)
print("Select train.csv, val.csv, test.csv, raw.csv (multi-select) from artifacts/data/:")
uploaded = files.upload()
for name in uploaded:
    shutil.move(name, f"artifacts/data/{name}")
print("Saved to artifacts/data/:", os.listdir("artifacts/data"))


In [ ]:
# --- Option B: Google Drive (alternative to the upload cell above) ---
# from google.colab import drive
# drive.mount('/content/drive')
# PROJECT_DIR = "/content/drive/MyDrive/DAMO"   # <-- adjust to where you uploaded the project
# import shutil
# shutil.copytree(f"{PROJECT_DIR}/artifacts/data", "artifacts/data", dirs_exist_ok=True)


In [ ]:
import pandas as pd

TARGET = "charges"
NUMERIC_FEATURES = ["age", "bmi", "children"]
CATEGORICAL_FEATURES = [
    "gender", "smoker", "region", "medical_history", "family_medical_history",
    "exercise_frequency", "occupation", "coverage_level",
]

train_df = pd.read_csv("artifacts/data/train.csv")
val_df = pd.read_csv("artifacts/data/val.csv")
test_df = pd.read_csv("artifacts/data/test.csv")
raw_df = pd.read_csv("artifacts/data/raw.csv")

print("train:", train_df.shape, "val:", val_df.shape, "test:", test_df.shape)
train_df.head()


## 3. Preprocessing — identical `ColumnTransformer` to `src/components/data_transformation.py`

In [ ]:
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
import joblib, os

numeric_pipeline = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler()),
])
categorical_pipeline = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore", sparse_output=False)),
])
preprocessor = ColumnTransformer(
    transformers=[
        ("numeric", numeric_pipeline, NUMERIC_FEATURES),
        ("categorical", categorical_pipeline, CATEGORICAL_FEATURES),
    ],
    remainder="drop",
)

X_train, y_train = train_df.drop(columns=[TARGET]), train_df[TARGET].values
X_val, y_val = val_df.drop(columns=[TARGET]), val_df[TARGET].values
X_test, y_test = test_df.drop(columns=[TARGET]), test_df[TARGET].values

X_train_t = preprocessor.fit_transform(X_train)
X_val_t = preprocessor.transform(X_val)
X_test_t = preprocessor.transform(X_test)
feature_names = preprocessor.get_feature_names_out().tolist()

os.makedirs("artifacts/preprocessing", exist_ok=True)
joblib.dump(preprocessor, "artifacts/preprocessing/preprocessor.pkl")
print("Train:", X_train_t.shape, "Val:", X_val_t.shape, "Test:", X_test_t.shape)


## 4. GPU-tuned XGBoost hyperparameter search

Hyperparameters below are sized for a free-tier **T4 (16GB)** GPU on this
dataset (~70k training rows × ~30 one-hot columns — small enough that GPU
memory is never the constraint; search breadth is). If you're on an A100/L4
in Colab Pro, you can safely push `n_iter` to 60–100 and widen the grid
further with no code changes.

Key GPU-specific settings:
- `device="cuda"`, `tree_method="hist"` — the modern (XGBoost ≥2.0) GPU path.
- `RandomizedSearchCV(n_jobs=1)` — **important**: a single GPU can't be
  safely time-shared across multiple parallel CV worker processes, so we
  keep the search sequential and let XGBoost use the GPU for each fit
  instead of trying to parallelize across folds.


In [ ]:
import time
from sklearn.model_selection import RandomizedSearchCV, cross_val_score
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import numpy as np

device = "cuda" if GPU_OK else "cpu"
print("Training XGBoost on device:", device)

xgb_param_grid = {
    "n_estimators": [200, 400, 600, 800],
    "max_depth": [3, 4, 5, 6, 8],
    "learning_rate": [0.01, 0.03, 0.05, 0.08, 0.1],
    "subsample": [0.7, 0.8, 0.9, 1.0],
    "colsample_bytree": [0.7, 0.8, 0.9, 1.0],
    "min_child_weight": [1, 3, 5],
    "reg_lambda": [0.5, 1.0, 2.0, 5.0],
}

base_estimator = xgb.XGBRegressor(
    random_state=42,
    verbosity=0,
    device=device,
    tree_method="hist",
)

search = RandomizedSearchCV(
    estimator=base_estimator,
    param_distributions=xgb_param_grid,
    n_iter=40,              # raise to 60-100 on a bigger GPU (A100/L4)
    scoring="r2",
    cv=5,
    n_jobs=1,                # keep sequential — single shared GPU
    random_state=42,
    refit=True,
    verbose=1,
)

start = time.time()
search.fit(X_train_t, y_train)
train_time = time.time() - start

best_xgb = search.best_estimator_
print(f"Best params: {search.best_params_}")
print(f"Search time: {train_time:.1f}s")


## 5. CPU baselines for comparison (cheap — seconds, no GPU needed)

In [ ]:
from sklearn.linear_model import LinearRegression, Ridge

results = []

def evaluate_on_val(name, model, t):
    y_val_pred = model.predict(X_val_t)
    cv_scores = cross_val_score(model, X_train_t, y_train, cv=5, scoring="r2", n_jobs=-1)
    row = {
        "model": name,
        "val_r2": r2_score(y_val, y_val_pred),
        "val_mae": mean_absolute_error(y_val, y_val_pred),
        "val_rmse": float(np.sqrt(mean_squared_error(y_val, y_val_pred))),
        "cv_mean_r2": float(cv_scores.mean()),
        "cv_std_r2": float(cv_scores.std()),
        "train_time_sec": t,
    }
    results.append(row)
    print(f"{name:16s} val_r2={row['val_r2']:.4f}  val_rmse={row['val_rmse']:.2f}  time={t:.1f}s")
    return row

t0 = time.time(); lr = LinearRegression().fit(X_train_t, y_train); t_lr = time.time() - t0
evaluate_on_val("LinearRegression", lr, t_lr)

t0 = time.time(); ridge = Ridge(alpha=1.0).fit(X_train_t, y_train); t_ridge = time.time() - t0
evaluate_on_val("Ridge", ridge, t_ridge)

evaluate_on_val("XGBoost_GPU" if device == "cuda" else "XGBoost_CPU", best_xgb, train_time)

comparison_df = pd.DataFrame(results).sort_values("val_r2", ascending=False).reset_index(drop=True)
comparison_df


## 6. Pick champion model, evaluate on held-out test set

In [ ]:
import json

trained_models = {"LinearRegression": lr, "Ridge": ridge,
                   ("XGBoost_GPU" if device == "cuda" else "XGBoost_CPU"): best_xgb}

champion_name = comparison_df.iloc[0]["model"]
champion_model = trained_models[champion_name]
print("Champion model:", champion_name)

y_test_pred = champion_model.predict(X_test_t)
test_metrics = {
    "model_name": champion_name,
    "r2": float(r2_score(y_test, y_test_pred)),
    "mae": float(mean_absolute_error(y_test, y_test_pred)),
    "rmse": float(np.sqrt(mean_squared_error(y_test, y_test_pred))),
}
print(test_metrics)


## 7. Diagnostic plots (same set the local `ModelEvaluation` component produces)

In [ ]:
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style="whitegrid")
os.makedirs("artifacts/evaluation", exist_ok=True)

def save_fig(fig, name):
    path = f"artifacts/evaluation/{name}"
    fig.savefig(path, dpi=120, bbox_inches="tight")
    plt.close(fig)

# Actual vs predicted
fig, ax = plt.subplots(figsize=(7, 6))
ax.scatter(y_test, y_test_pred, alpha=0.35, s=14, color="#2563eb")
lims = [min(y_test.min(), y_test_pred.min()), max(y_test.max(), y_test_pred.max())]
ax.plot(lims, lims, "r--", lw=2, label="Perfect prediction")
ax.set_xlabel("Actual Insurance Charges ($)"); ax.set_ylabel("Predicted Insurance Charges ($)")
ax.set_title("Actual vs Predicted"); ax.legend()
save_fig(fig, "actual_vs_predicted.png")

# Residuals
residuals = y_test - y_test_pred
fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))
axes[0].scatter(y_test_pred, residuals, alpha=0.35, s=14, color="#16a34a")
axes[0].axhline(0, color="r", linestyle="--")
axes[0].set_xlabel("Predicted Value"); axes[0].set_ylabel("Residual"); axes[0].set_title("Residuals vs Predicted")
sns.histplot(residuals, kde=True, ax=axes[1], color="#f59e0b"); axes[1].set_title("Residual Distribution")
save_fig(fig, "residuals.png")

# Feature importance (tree models only)
top_features = {}
if hasattr(champion_model, "feature_importances_"):
    imp = pd.Series(champion_model.feature_importances_, index=feature_names).sort_values(ascending=False).head(15)
    fig, ax = plt.subplots(figsize=(7, 6))
    sns.barplot(x=imp.values, y=imp.index, hue=imp.index, ax=ax, palette="viridis", legend=False)
    ax.set_title("Top Feature Importances"); ax.set_xlabel("Importance")
    save_fig(fig, "feature_importance.png")
    top_features = imp.to_dict()
elif hasattr(champion_model, "coef_"):
    imp = pd.Series(np.abs(champion_model.coef_), index=feature_names).sort_values(ascending=False).head(15)
    fig, ax = plt.subplots(figsize=(7, 6))
    sns.barplot(x=imp.values, y=imp.index, hue=imp.index, ax=ax, palette="viridis", legend=False)
    ax.set_title("Top Feature Importances (|coef|)"); ax.set_xlabel("Importance")
    save_fig(fig, "feature_importance.png")
    top_features = imp.to_dict()

# Correlation heatmap (raw numeric features)
numeric_df = raw_df.select_dtypes(include=[np.number])
fig, ax = plt.subplots(figsize=(8, 7))
sns.heatmap(numeric_df.corr(), cmap="coolwarm", center=0, annot=True, fmt=".2f", ax=ax)
ax.set_title("Correlation Heatmap")
save_fig(fig, "correlation_heatmap.png")

test_metrics["top_features"] = top_features
print("Plots saved to artifacts/evaluation/")


## 8. Save artifacts in the exact layout the local project expects

In [ ]:
os.makedirs("artifacts/models", exist_ok=True)

joblib.dump(champion_model, "artifacts/models/best_model.pkl")

with open("artifacts/models/model_metadata.json", "w") as f:
    json.dump({
        "best_model_name": champion_name,
        "val_r2": float(comparison_df.iloc[0]["val_r2"]),
        "val_rmse": float(comparison_df.iloc[0]["val_rmse"]),
    }, f, indent=4, default=str)

comparison_df.to_csv("artifacts/evaluation/model_comparison.csv", index=False)

with open("artifacts/evaluation/evaluation_report.json", "w") as f:
    json.dump(test_metrics, f, indent=4, default=str)

print("Test R2:", test_metrics["r2"], "| Test RMSE:", test_metrics["rmse"])
print("Champion:", champion_name)


## 9. Download everything

Zips `artifacts/models/`, `artifacts/preprocessing/`, and `artifacts/evaluation/`
so you can download it and overwrite the corresponding folders in your local
`DAMO/` project — `streamlit_app/app.py` and `src/pipeline/predict_pipeline.py`
will pick up the new model automatically, no code changes needed.


In [ ]:
import shutil
shutil.make_archive("gpu_trained_artifacts", "zip", ".", "artifacts")
from google.colab import files
files.download("gpu_trained_artifacts.zip")
